# Gradient-Boosted Ensemble (LightGBM)
This notebook trains our primary model: a LightGBM Regressor using GroupKFold cross-validation. We predict the change in True Vertical Thickness (`dTVT`) after the Prediction Start point.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold, ParameterGrid
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('../src'))
from data_loader import get_all_well_ids, load_well_data
from features import extract_features_for_well

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

In [1]:
DATA_DIR = '../data'
well_ids = get_all_well_ids(DATA_DIR, split='train')

# Using the first 100 wells for demonstration. 
# Remove the slicing to train on all 773 wells for the final submission.
num_wells_to_use = 100
selected_wells = well_ids[:num_wells_to_use]
print(f"Processing {len(selected_wells)} wells...")

all_features = []
for wid in selected_wells:
    horz_df, type_df = load_well_data(wid, DATA_DIR, split='train')
    df_feats = extract_features_for_well(wid, horz_df, type_df, is_train=True)
    if len(df_feats) > 0:
        all_features.append(df_feats)

dataset = pd.concat(all_features, ignore_index=True)
print(f"Dataset Shape: {dataset.shape}")

NameError: name 'get_all_well_ids' is not defined

In [2]:
# Prepare features and target
dataset['target_dTVT'] = dataset['TVT'] - dataset['ps_TVT']

# LightGBM can handle NaNs natively, so we don't strictly need to fillna
# However, we ensure it's clean for comparison with Ridge.
features = ['delta_Z', 'delta_MD', 'dist_XY', 'GR', 'inclination_proxy']

X = dataset[features]
y = dataset['target_dTVT']
groups = dataset['well_id']

NameError: name 'dataset' is not defined

In [3]:
# Hyperparameter tuning + Cross-Validation
gkf = GroupKFold(n_splits=5)

# Base parameters
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'n_estimators': 300,
    'num_leaves': 31,
    'random_state': 42,
    'n_jobs': -1
}

rmse_scores = []
models = []

print("Starting LightGBM GroupKFold CV...")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params)
    
    # Note: LightGBM early stopping requires passing eval_set
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Predict
    preds_val = model.predict(X_val)
    
    # Reconstruct absolute TVT to get actual RMSE
    ps_TVT_val = dataset.iloc[val_idx]['ps_TVT'].values
    pred_abs_TVT = ps_TVT_val + preds_val
    true_abs_TVT = dataset.iloc[val_idx]['TVT'].values
    
    score = root_mean_squared_error(true_abs_TVT, pred_abs_TVT)
    rmse_scores.append(score)
    models.append(model)
    
    print(f"Fold {fold+1} RMSE: {score:.2f} feet")

print(f"\nAverage CV RMSE: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f} feet")

NameError: name 'GroupKFold' is not defined

In [4]:
# Plot Feature Importances from the last fold model
best_model = models[-1]
lgb.plot_importance(best_model, importance_type='gain', max_num_features=10, title='LightGBM Feature Importance (Gain)')
plt.show()

NameError: name 'models' is not defined